# Fashion MNIST Training Experiments - Session 3
## Training Neural Networks

**Goal:** Achieve >85% accuracy through systematic training experiments

### Experiments:
1. Batch size variation
2. Learning rate tuning
3. Optimizer comparison
4. Early stopping

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Load Fashion MNIST
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_test = X_test.reshape(-1, 784).astype('float32') / 255.0
y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat = keras.utils.to_categorical(y_test, 10)

class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
print(f'Classes: {class_names}')

In [ ]:
def build_model(learning_rate=0.001):
    model = Sequential([
        Dense(128, activation='relu', input_shape=(784,)),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(10, activation='softmax')
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [ ]:
# Experiment 1: Batch size
print('Experiment 1: Batch Size')
batch_sizes = [32, 64, 128, 256]
results_batch = {}

for bs in batch_sizes:
    print(f'  Testing batch_size={bs}...')
    model = build_model()
    history = model.fit(X_train, y_train_cat, validation_split=0.2, epochs=20, batch_size=bs, verbose=0)
    test_acc = model.evaluate(X_test, y_test_cat, verbose=0)[1]
    results_batch[bs] = test_acc
    print(f'    Accuracy: {test_acc*100:.2f}%')

print(f'\nBest batch_size: {max(results_batch, key=results_batch.get)} ({max(results_batch.values())*100:.2f}%)')

In [ ]:
# Experiment 2: Learning rate
print('\nExperiment 2: Learning Rate')
lrs = [0.0001, 0.001, 0.01]
results_lr = {}

for lr in lrs:
    print(f'  Testing lr={lr}...')
    model = build_model(learning_rate=lr)
    history = model.fit(X_train, y_train_cat, validation_split=0.2, epochs=20, batch_size=128, verbose=0)
    test_acc = model.evaluate(X_test, y_test_cat, verbose=0)[1]
    results_lr[lr] = test_acc
    print(f'    Accuracy: {test_acc*100:.2f}%')

print(f'\nBest learning_rate: {max(results_lr, key=results_lr.get)} ({max(results_lr.values())*100:.2f}%)')

In [ ]:
# Final optimized model with early stopping
print('\nTraining final optimized model...')
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

final_model = build_model(learning_rate=0.001)
history = final_model.fit(X_train, y_train_cat, validation_split=0.2, epochs=50,
                         batch_size=128, callbacks=[early_stop], verbose=1)

final_acc = final_model.evaluate(X_test, y_test_cat, verbose=0)[1]
print(f'\n✅ Final Test Accuracy: {final_acc*100:.2f}%')

if final_acc >= 0.85:
    print('🎉 SUCCESS! Achieved >85% accuracy!')
else:
    print(f'⚠️ Close! Target: 85%, Current: {final_acc*100:.2f}%')

In [ ]:
# Save model
final_model.save('fashion_mnist_model.h5')
print('✅ Model saved!')